# 🏦 Loan Risk Assessment — German Credit Dataset
**Upload `german_credit_data.csv` when prompted, then run all cells in order.**

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q pandas numpy scikit-learn matplotlib seaborn xgboost ipywidgets

In [ ]:
# ── Cell 2: Upload dataset ────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # upload german_credit_data.csv here

In [ ]:
# ── Cell 3: Load & preview data ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('german_credit_data.csv', index_col=0)
print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# ── Cell 4: Exploratory Data Analysis ────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('German Credit Dataset — EDA', fontsize=16, fontweight='bold', y=1.01)

# Risk distribution
risk_counts = df['Risk'].value_counts()
colors = ['#2d6247', '#8b2020']
axes[0,0].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%',
              colors=colors, startangle=90, textprops={'fontsize':12})
axes[0,0].set_title('Risk Distribution')

# Age distribution by risk
for risk, color in zip(['good','bad'], colors):
    subset = df[df['Risk']==risk]['Age']
    axes[0,1].hist(subset, bins=20, alpha=0.6, color=color, label=risk)
axes[0,1].set_title('Age by Risk'); axes[0,1].legend(); axes[0,1].set_xlabel('Age')

# Credit amount by risk
df.boxplot(column='Credit amount', by='Risk', ax=axes[0,2], 
           boxprops=dict(color='#1a3a2a'), medianprops=dict(color='#2d6247', linewidth=2))
axes[0,2].set_title('Credit Amount by Risk'); axes[0,2].set_xlabel('')
plt.sca(axes[0,2]); plt.title('Credit Amount by Risk')

# Duration by risk
df.boxplot(column='Duration', by='Risk', ax=axes[1,0],
           boxprops=dict(color='#1a3a2a'), medianprops=dict(color='#2d6247', linewidth=2))
axes[1,0].set_title('Duration by Risk'); axes[1,0].set_xlabel('')
plt.sca(axes[1,0]); plt.title('Duration by Risk')

# Housing vs Risk
housing_risk = pd.crosstab(df['Housing'], df['Risk'], normalize='index') * 100
housing_risk.plot(kind='bar', ax=axes[1,1], color=colors, edgecolor='none', rot=0)
axes[1,1].set_title('Housing vs Risk (%)'); axes[1,1].set_ylabel('%'); axes[1,1].legend()

# Purpose vs Risk
purpose_risk = pd.crosstab(df['Purpose'], df['Risk'], normalize='index') * 100
purpose_risk['bad'].sort_values().plot(kind='barh', ax=axes[1,2], color='#8b2020', edgecolor='none')
axes[1,2].set_title('% Bad Risk by Purpose'); axes[1,2].set_xlabel('% Bad')
axes[1,2].axvline(30, color='#2d6247', linestyle='--', linewidth=1, label='30% baseline')
axes[1,2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA complete.')

In [ ]:
# ── Cell 5: Preprocessing ─────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df2 = df.copy()

# Fill NA in account columns with 'unknown'
df2['Saving accounts'] = df2['Saving accounts'].fillna('unknown')
df2['Checking account'] = df2['Checking account'].fillna('unknown')

# Encode ordinal features
saving_order = {'unknown':0, 'little':1, 'moderate':2, 'quite rich':3, 'rich':4}
checking_order = {'unknown':0, 'little':1, 'moderate':2, 'rich':3}
df2['Saving accounts'] = df2['Saving accounts'].map(saving_order)
df2['Checking account'] = df2['Checking account'].map(checking_order)

# One-hot encode remaining categoricals
cat_cols = ['Sex', 'Housing', 'Purpose']
df2 = pd.get_dummies(df2, columns=cat_cols, drop_first=True)

# Target encoding
df2['Risk'] = (df2['Risk'] == 'bad').astype(int)   # 1 = bad, 0 = good

X = df2.drop('Risk', axis=1)
y = df2['Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Bad risk rate — train: {y_train.mean():.2%}, test: {y_test.mean():.2%}')
print(f'\nFeatures ({len(X.columns)}):', list(X.columns))

In [ ]:
# ── Cell 6: Train multiple models & compare ───────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              classification_report, confusion_matrix)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, use_label_encoder=False,
                                         eval_metric='logloss', random_state=42),
}

results = {}
for name, model in models.items():
    # Logistic Regression needs scaled features
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test
    model.fit(X_tr, y_train)
    preds  = model.predict(X_te)
    proba  = model.predict_proba(X_te)[:,1]
    acc    = accuracy_score(y_test, preds)
    auc    = roc_auc_score(y_test, proba)
    results[name] = {'model': model, 'acc': acc, 'auc': auc,
                     'preds': preds, 'proba': proba}
    print(f'{name:<25}  Accuracy: {acc:.3f}   AUC: {auc:.3f}')

best_name = max(results, key=lambda k: results[k]['auc'])
print(f'\n★ Best model by AUC: {best_name} ({results[best_name]["auc"]:.3f})')

In [ ]:
# ── Cell 7: Detailed evaluation of best model ─────────────────────────────────
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

best = results[best_name]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Model Evaluation — {best_name}', fontsize=14, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(y_test, best['preds'])
disp = ConfusionMatrixDisplay(cm, display_labels=['Good (0)', 'Bad (1)'])
disp.plot(ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title('Confusion Matrix')

# ROC curves for all models
for name, res in results.items():
    X_te = X_test_sc if name == 'Logistic Regression' else X_test
    RocCurveDisplay.from_predictions(
        y_test, res['proba'], name=f"{name} ({res['auc']:.2f})", ax=axes[1])
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_title('ROC Curves (AUC)')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClassification Report — {best_name}:')
print(classification_report(y_test, best['preds'], target_names=['Good','Bad']))

In [ ]:
# ── Cell 8: Feature importance ────────────────────────────────────────────────
best_model = best['model']

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances})
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_[0])
    feat_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances})
else:
    print('Feature importances not available for this model type.')
    feat_df = None

if feat_df is not None:
    feat_df = feat_df.sort_values('Importance', ascending=True).tail(15)
    plt.figure(figsize=(9, 6))
    colors_bar = ['#2d6247' if i >= len(feat_df)-5 else '#9abfaf' for i in range(len(feat_df))]
    plt.barh(feat_df['Feature'], feat_df['Importance'], color=colors_bar, edgecolor='none')
    plt.title(f'Top Feature Importances — {best_name}', fontsize=13, fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(feat_df.sort_values('Importance', ascending=False).to_string(index=False))

In [ ]:
# ── Cell 9: Risk Assessment Function ─────────────────────────────────────────
# Uses the best model to score a single applicant.

def assess_loan_risk(age, sex, job, housing, saving_accounts,
                     checking_account, credit_amount, duration, purpose):
    """
    Predict loan risk for a single applicant.

    Parameters
    ----------
    age              : int
    sex              : 'male' | 'female'
    job              : 0 | 1 | 2 | 3
    housing          : 'own' | 'rent' | 'free'
    saving_accounts  : 'unknown' | 'little' | 'moderate' | 'quite rich' | 'rich'
    checking_account : 'unknown' | 'little' | 'moderate' | 'rich'
    credit_amount    : int/float  (€)
    duration         : int  (months)
    purpose          : 'car' | 'furniture/equipment' | 'radio/TV' |
                       'domestic appliances' | 'repairs' | 'education' |
                       'business' | 'vacation/others'

    Returns
    -------
    dict with verdict, probability, and risk score
    """
    inp = pd.DataFrame([{
        'Age': age, 'Sex': sex, 'Job': job, 'Housing': housing,
        'Saving accounts': saving_accounts,
        'Checking account': checking_account,
        'Credit amount': credit_amount,
        'Duration': duration, 'Purpose': purpose
    }])

    # Ordinal encoding
    inp['Saving accounts']  = inp['Saving accounts'].map(saving_order).fillna(0)
    inp['Checking account'] = inp['Checking account'].map(checking_order).fillna(0)

    # One-hot
    inp = pd.get_dummies(inp, columns=['Sex','Housing','Purpose'], drop_first=True)

    # Align to training columns
    for col in X_train.columns:
        if col not in inp.columns:
            inp[col] = 0
    inp = inp[X_train.columns]

    # Scale if needed
    if best_name == 'Logistic Regression':
        inp_ready = scaler.transform(inp)
    else:
        inp_ready = inp

    prob_bad  = best_model.predict_proba(inp_ready)[0][1]
    prob_good = 1 - prob_bad
    risk_score = round(prob_bad * 100, 1)

    if prob_bad < 0.35:
        verdict = '✅  LOW RISK  — Approve'
    elif prob_bad < 0.60:
        verdict = '⚠️  MODERATE RISK — Manual Review'
    else:
        verdict = '❌  HIGH RISK — Decline'

    return {
        'verdict':        verdict,
        'prob_bad':       round(prob_bad, 4),
        'prob_good':      round(prob_good, 4),
        'risk_score_pct': risk_score,
        'model_used':     best_name,
    }

print('assess_loan_risk() is ready.  See Cell 10 for examples.')

In [ ]:
# ── Cell 10: Try example applicants ───────────────────────────────────────────

examples = [
    dict(age=35, sex='male',   job=2, housing='own',  saving_accounts='moderate',
         checking_account='moderate', credit_amount=4000, duration=24, purpose='car'),
    dict(age=22, sex='female', job=1, housing='rent', saving_accounts='little',
         checking_account='little',   credit_amount=9000, duration=60, purpose='vacation/others'),
    dict(age=50, sex='male',   job=3, housing='own',  saving_accounts='rich',
         checking_account='rich',     credit_amount=2000, duration=12, purpose='education'),
]

for i, ex in enumerate(examples, 1):
    result = assess_loan_risk(**ex)
    print(f"\n── Applicant {i} ─────────────────────────────────")
    for k, v in ex.items():
        print(f"  {k:<22}: {v}")
    print(f"  {'→ Verdict':<22}: {result['verdict']}")
    print(f"  {'→ Bad risk prob':<22}: {result['prob_bad']:.1%}")
    print(f"  {'→ Risk score':<22}: {result['risk_score_pct']}/100")

In [ ]:
# ── Cell 11: Interactive widget (Colab UI) ────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output

style  = {'description_width': '180px'}
layout = widgets.Layout(width='420px')

w_age      = widgets.IntSlider(value=35, min=18, max=80, step=1,
                               description='Age', style=style, layout=layout)
w_sex      = widgets.Dropdown(options=['male','female'],
                               description='Sex', style=style, layout=layout)
w_job      = widgets.Dropdown(options=[0,1,2,3], value=2,
                               description='Job level (0-3)', style=style, layout=layout)
w_housing  = widgets.Dropdown(options=['own','rent','free'],
                               description='Housing', style=style, layout=layout)
w_saving   = widgets.Dropdown(options=['unknown','little','moderate','quite rich','rich'],
                               description='Saving accounts', style=style, layout=layout)
w_checking = widgets.Dropdown(options=['unknown','little','moderate','rich'],
                               description='Checking account', style=style, layout=layout)
w_amount   = widgets.IntSlider(value=3000, min=250, max=18500, step=250,
                               description='Credit amount (€)', style=style, layout=layout)
w_duration = widgets.IntSlider(value=24, min=4, max=72, step=1,
                               description='Duration (months)', style=style, layout=layout)
w_purpose  = widgets.Dropdown(
    options=['car','furniture/equipment','radio/TV','domestic appliances',
             'repairs','education','business','vacation/others'],
    description='Purpose', style=style, layout=layout)
btn    = widgets.Button(description='  Assess Risk', button_style='success',
                        icon='search', layout=widgets.Layout(width='180px', height='38px'))
output = widgets.Output()

def on_click(_):
    with output:
        clear_output(wait=True)
        r = assess_loan_risk(
            age=w_age.value, sex=w_sex.value, job=w_job.value,
            housing=w_housing.value, saving_accounts=w_saving.value,
            checking_account=w_checking.value, credit_amount=w_amount.value,
            duration=w_duration.value, purpose=w_purpose.value)
        print('\n' + '═'*48)
        print(f"  RESULT")
        print('═'*48)
        print(f"  {r['verdict']}")
        print(f"  Risk score  : {r['risk_score_pct']}/100")
        print(f"  P(bad)      : {r['prob_bad']:.1%}")
        print(f"  P(good)     : {r['prob_good']:.1%}")
        print(f"  Model used  : {r['model_used']}")
        print('═'*48)

btn.on_click(on_click)

print('\n🏦  Loan Risk Assessment — Interactive Widget\n')
display(widgets.VBox([
    w_age, w_sex, w_job, w_housing,
    w_saving, w_checking, w_amount, w_duration, w_purpose,
    btn, output
]))

In [ ]:
# ── Cell 12: Save best model ──────────────────────────────────────────────────
import pickle

artifact = {
    'model':         best_model,
    'scaler':        scaler,
    'feature_cols':  list(X_train.columns),
    'saving_order':  saving_order,
    'checking_order':checking_order,
    'best_name':     best_name,
}
with open('loan_risk_model.pkl','wb') as f:
    pickle.dump(artifact, f)
print('Model saved to loan_risk_model.pkl')

# Download it
files.download('loan_risk_model.pkl')